# Impact of Plant Growth Regulator (PGR) on barley varieties

## Objective

Testing the effect of Plant Growth Regulator (PGR) on 3 varieties of Barley with 4 replicates using the split plot design. 9 trt x 3 variety x 4 rep

## 1- Load libraries

In [45]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(tidymodels)
  library(readxl)
  library(janitor)
  library(plotly)
  library(corrr)
  library(sjPlot)
  library(GGally)
  library(ggpubr)
  library(patchwork)
  library(agricolae)
  library(lmerTest)
  library(openxlsx)
})

## 2- Load Custom functions

In [46]:
source("function.R")

## 3- Load data

In [50]:
data <- readxl::read_excel(
  "data/All Year 28 R Barley trial data.xlsx",
  sheet = "Overall_2",
  na = c("", "NA", "N/A", ".", "..", "-", "—")
) %>%
  clean_names() %>%
  filter(!(location == "Falher" & year == 2024)) %>%
  
  # Convert ALL character columns that should be numeric
  mutate(
    across(
      c(
        yield_adj_to_13_5_percent,
        yield_adj_to_13_5_percent_bu,
        avg_height_at_phys_maturity,
        lodging_index_1,
        lodging_index_2,
        lodging_index_3,
        tkw_avg,
        ndf_percent_dm,
        iv_d_ec_percent_dm,
        starch_percent_dm,
        test_weight_kg_h_l,
        protein_percent_dm,
        dtm,
        grain_moisture_percent
      ),
      ~ if (is.character(.x)) readr::parse_number(.x) else as.numeric(.x)
    )
  ) %>%
  
  mutate(
    cultivar = case_when(
      cultivar == "TR18647" ~ "AB_Hague",
      TRUE ~ cultivar
    )
  )

## 4- Data Cleaning

In [49]:
cleaned_data_temp <-
  data %>% 
  mutate(lodging_avg = (lodging_index_1 + lodging_index_2 + lodging_index_3) /3) %>% 
  rename(tkw = tkw_avg_adjusted_to_13_5_percent_moisture,
         ndf_percent_af = ndf_percent_dm,
         plant_height_maturity_cm = emerge_avg_plants_m2,
         yield_bu_acre = yield_adj_to_13_5_percent_bu
         ) %>% 
  select("year", "location", "cultivar", "pgr_trt_name",  "yield_bu_acre",
  "lodging_index_3",
  "protein_percent_dm",
  "avg_height_at_phys_maturity",
  "tkw",
  "test_weight_kg_h_l",
  "dtm",
  "ndf_percent_af",
  "iv_d_ec_percent_dm",
  "grain_moisture_percent",
   "starch_percent_dm",
  "yield_adj_to_13_5_percent",
  "lodging_avg",
  "lodging_index_1",
  "lodging_index_2",
  "lodging_index_3")

# Display the first 6 row of the dataset
head(cleaned_data_temp)

year,location,cultivar,pgr_trt_name,yield_bu_acre,lodging_index_3,protein_percent_dm,avg_height_at_phys_maturity,tkw,test_weight_kg_h_l,dtm,ndf_percent_af,iv_d_ec_percent_dm,grain_moisture_percent,starch_percent_dm,yield_adj_to_13_5_percent,lodging_avg,lodging_index_1,lodging_index_2
<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2022,Vermilion,CDC_Austenson,No_PGR,135.5724,3.333333,12.35137,72.33333,46.84555,66.13,88.55556,16.94731,3484.369,11.91,61.80709,7292.860,2.222222,0,3.333333
2022,Vermilion,CDC_Austenson,M62.5_21-24,127.2140,0.000000,12.16454,74.00000,45.59578,65.01,91.91429,16.99448,3492.453,11.37,61.71610,6843.238,0.000000,0,0.000000
2022,Vermilion,CDC_Austenson,M62.5_30-32,117.1409,0.000000,11.94270,69.66667,49.55971,69.78,92.14286,17.04296,3429.073,11.61,60.99495,6301.371,0.000000,0,0.000000
2022,Vermilion,CDC_Austenson,M62.5_37,175.9922,0.000000,11.90405,71.66667,46.57075,65.25,91.68687,16.27136,3398.832,14.29,61.71217,9467.169,0.000000,0,0.000000
2022,Vermilion,CDC_Austenson,M125_30-32,138.3530,0.000000,12.28261,71.00000,48.57156,66.56,94.79487,17.06522,3464.130,12.47,61.19565,7442.436,0.000000,0,0.000000
2022,Vermilion,CDC_Austenson,M125_37,156.1708,0.000000,12.39285,76.66667,46.64879,67.16,93.10989,16.46905,3407.598,12.28,60.91529,8400.912,0.000000,0,0.000000


## 5- Feature Engineering

In [53]:
# Create a recipe
rec_1 <- recipe(yield_adj_to_13_5_percent ~ ., data = cleaned_data_temp) %>%
  step_impute_knn(all_predictors(), neighbors = 20)  # use k = 5

# Prep the recipe
rec_prepped_1 <- prep(rec_1)

# Apply the imputation to the full dataset
cleaned_data_imputed_1 <- bake(rec_prepped_1, new_data = NULL) %>% 
  select(-yield_adj_to_13_5_percent)

In [54]:
# Create a recipe
rec_2 <- recipe(yield_bu_acre ~ ., data = cleaned_data_temp) %>%
  step_impute_knn(all_predictors(), neighbors = 20)  # use k = 5

# Prep the recipe
rec_prepped_2 <- prep(rec_2)

# Apply the imputation to the full dataset
cleaned_data_imputed_2 <- bake(rec_prepped_2, new_data = NULL) %>% 
  select(-yield_bu_acre)

In [81]:
cleaned_data_imputed <- cleaned_data_imputed_1 %>% 
  left_join(cleaned_data_imputed_2) %>% clean_names()
head(cleaned_data_imputed)

Joining with `by = join_by(year, location, cultivar, pgr_trt_name,
lodging_index_3, protein_percent_dm, avg_height_at_phys_maturity, tkw,
test_weight_kg_h_l, dtm, ndf_percent_af, iv_d_ec_percent_dm,
grain_moisture_percent, starch_percent_dm, lodging_avg, lodging_index_1,
lodging_index_2)`


year,location,cultivar,pgr_trt_name,yield_bu_acre,lodging_index_3,protein_percent_dm,avg_height_at_phys_maturity,tkw,test_weight_kg_h_l,dtm,ndf_percent_af,iv_d_ec_percent_dm,grain_moisture_percent,starch_percent_dm,lodging_avg,lodging_index_1,lodging_index_2,yield_adj_to_13_5_percent
<dbl>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2022,Vermilion,CDC_Austenson,No_PGR,135.5724,3.333333,12.35137,72.33333,46.84555,66.13,88.55556,16.94731,3484.369,11.91,61.80709,2.222222,0,3.333333,7292.860
2022,Vermilion,CDC_Austenson,M62.5_21-24,127.2140,0.000000,12.16454,74.00000,45.59578,65.01,91.91429,16.99448,3492.453,11.37,61.71610,0.000000,0,0.000000,6843.238
2022,Vermilion,CDC_Austenson,M62.5_30-32,117.1409,0.000000,11.94270,69.66667,49.55971,69.78,92.14286,17.04296,3429.073,11.61,60.99495,0.000000,0,0.000000,6301.371
2022,Vermilion,CDC_Austenson,M62.5_37,175.9922,0.000000,11.90405,71.66667,46.57075,65.25,91.68687,16.27136,3398.832,14.29,61.71217,0.000000,0,0.000000,9467.169
2022,Vermilion,CDC_Austenson,M125_30-32,138.3530,0.000000,12.28261,71.00000,48.57156,66.56,94.79487,17.06522,3464.130,12.47,61.19565,0.000000,0,0.000000,7442.436
2022,Vermilion,CDC_Austenson,M125_37,156.1708,0.000000,12.39285,76.66667,46.64879,67.16,93.10989,16.46905,3407.598,12.28,60.91529,0.000000,0,0.000000,8400.912


## 5.1- Save data

In [116]:
saveRDS(cleaned_data_imputed, "data/cleaned_data_imputed.rds")

## 7- Run statistical test by treatment & location & Year

### 7.1- Automated LSD and CV Analysis Pipeline for PGR Treatment Trials Across Locations, Years, and Cultivars

In [ ]:
# Define the standard treatment order for consistent output
trt_order <- c(
  "No_PGR",
  "M62.5_21-24",
  "M62.5_30-32",
  "M62.5_37",
  "M125_30-32",
  "M125_37",
  "M62.5_21-24 + M62.5_37",
  "M62.5_21-24 + M62.5_30-32",
  "M62.5_30-32 + M62.5_37"
)


# Run LSD and CV analysis for one location-year-cultivar combination
run_trait <- function(dat, location_, year_, cultivar_, outcome_col) {

  # Filter data and prepare treatment and outcome variables
  df <- dat %>%
    filter(location == location_,
           year == year_,
           cultivar == cultivar_) %>%
    transmute(
      location,
      year,
      cultivar,
      treatment = trimws(as.character(pgr_trt_name)),
      outcome   = as.numeric(.data[[outcome_col]])
    ) %>%
    mutate(treatment = factor(treatment, levels = trt_order))


  # Return empty results if no data are available
  if (nrow(df) == 0) {
    return(list(
      lsd = tibble(
        treatment = factor(levels = trt_order),
        mean = numeric(),
        SE = numeric(),
        groups = character()
      ),
      cv = tibble(sd = NA_real_, avg = NA_real_, cv = NA_real_)
    ))
  }


  # Calculate coefficient of variation
  cv_tbl <- df %>%
    summarise(
      sd  = sd(outcome, na.rm = TRUE),
      avg = mean(outcome, na.rm = TRUE),
      cv  = sd / avg
    )


  # Run LSD test and handle errors safely
  lsd_tbl <- tryCatch({

    out <- lsd_test_f2_01(df) %>% tibble::as_tibble()

    # Reapply treatment order
    out$treatment <- factor(
      trimws(as.character(out$treatment)),
      levels = trt_order
    )

    out

  }, error = function(e) {

    # Return empty table if LSD test fails
    tibble(
      treatment = factor(levels = trt_order),
      mean = numeric(),
      SE = numeric(),
      groups = character()
    )

  })


  # Return LSD and CV results
  list(lsd = lsd_tbl, cv = cv_tbl)
}


# Define all location-year-cultivar combinations to analyze
plan <- tibble::tribble(
  ~location,     ~year, ~cultivar,
  "Falher",      2022,  "CDC_Austenson",
  "Falher",      2022,  "AB_Hague",
  "Falher",      2022,  "Esma",
  "Falher",      2023,  "CDC_Austenson",
  "Falher",      2023,  "AB_Hague",
  "Falher",      2023,  "Esma",

  "Lethbridge",  2022,  "CDC_Austenson",
  "Lethbridge",  2022,  "AB_Hague",
  "Lethbridge",  2022,  "Esma",
  "Lethbridge",  2023,  "CDC_Austenson",
  "Lethbridge",  2023,  "AB_Hague",
  "Lethbridge",  2023,  "Esma",
  "Lethbridge",  2024,  "CDC_Austenson", 
  "Lethbridge",  2024,  "AB_Hague",
  "Lethbridge",  2024,  "Esma",

  "Vermilion",   2022,  "CDC_Austenson", 
  "Vermilion",   2022,  "AB_Hague",
  "Vermilion",   2022,  "Esma",
  "Vermilion",   2023,  "CDC_Austenson", 
  "Vermilion",   2023,  "AB_Hague",
  "Vermilion",   2023,  "Esma",
  "Vermilion",   2024,  "CDC_Austenson", 
  "Vermilion",   2024,  "AB_Hague",
  "Vermilion",   2024,  "Esma"
)


# Run analysis for one trait across all trials
run_all <- function(outcome_col,
                    trait_name = outcome_col,
                    digits = 2) {

  # Apply run_trait to each plan combination
  results <- plan %>%
    mutate(
      res = pmap(
        list(location, year, cultivar),
        ~ run_trait(cleaned_data_imputed, ..1, ..2, ..3, outcome_col)
      )
    )


  # Extract CV results
  cv_table <- results %>%
    transmute(location, year, cultivar, cv = map(res, "cv")) %>%
    unnest(cv)


  # Extract LSD results
  lsd_table <- results %>%
    transmute(location, year, cultivar, lsd = map(res, "lsd")) %>%
    unnest(lsd)


  # Merge results and format output table
  lsd_table %>%
    left_join(cv_table,
              by = c("location", "year", "cultivar")) %>%
    mutate(

      # Add trait name
      trait = trait_name,

      # Enforce treatment ordering
      treatment = factor(
        trimws(as.character(treatment)),
        levels = trt_order
      ),

      # Format mean, group, and SE for reporting
      mean_group_se = sprintf(
        paste0("%.", digits, "f %s \u00B1 %.", digits, "f"),
        mean,
        trimws(groups),
        SE
      )
    ) %>%
    
    # Sort results for presentation
    arrange(trait, location, year, cultivar, treatment)
}


### 7.2- Lodging

In [85]:
lodging_tbl <- run_all("lodging_index_3", "lodging")

write.xlsx(
  lodging_tbl,
  file = "statistics_tables/lodging_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.3- Yield kg/ha

In [86]:
yield_tbl   <- run_all("yield_adj_to_13_5_percent", "yield")   
write.xlsx(
  yield_tbl,
  file = "statistics_tables/yield_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.4- Yield Bu/ac

In [87]:
yield_bu_acre_tbl   <- run_all("yield_bu_acre", "yield_bu_acre")   
write.xlsx(
  yield_bu_acre_tbl,
  file = "statistics_tables/yield_bu_acre_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.5- TKW

In [89]:
tkw_tbl     <- run_all("tkw", "tkw")
write.xlsx(
  tkw_tbl,
  file = "statistics_tables/tkw_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.6- Protein

In [90]:
protein_tbl <- run_all("protein_percent_dm", "protein")  
write.xlsx(
  protein_tbl,
  file = "statistics_tables/protein_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.7- DTM

In [91]:
dtm_tbl <- run_all("dtm", "dtm")  
write.xlsx(
  dtm_tbl,
  file = "statistics_tables/dtm_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.8- Test Weight

In [92]:
test_weight_kg_h_l_tbl <- run_all("test_weight_kg_h_l", "test_weight_kg_h_l")  
write.xlsx(
  test_weight_kg_h_l_tbl,
  file = "statistics_tables/test_weight_kg_h_l_tbl_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.9- NDF

In [93]:
ndf_percent_af_tbl <- run_all("ndf_percent_af", "ndf_percent_af")  
write.xlsx(
  ndf_percent_af_tbl,
  file = "statistics_tables/ndf_percent_af_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.10- IV_DEC

In [94]:
iv_d_ec_percent_dm_tbl <- run_all("iv_d_ec_percent_dm", "iv_d_ec_percent_dm")  
write.xlsx(
  iv_d_ec_percent_dm_tbl,
  file = "statistics_tables/iv_d_ec_percent_dm_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.11- Average Plant Height at maturity

In [95]:
avg_height_at_phys_maturity_tbl <- run_all("avg_height_at_phys_maturity", "avg_height_at_phys_maturity")  
write.xlsx(
  avg_height_at_phys_maturity_tbl,
  file = "statistics_tables/avg_height_at_phys_maturity_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.12- Starch

In [96]:
starch_percent_dm_tbl <- run_all("starch_percent_dm", "starch_percent_dm")  
write.xlsx(
  starch_percent_dm_tbl,
  file = "statistics_tables/starch_percent_dm_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 7.13- Grain moisture content

In [97]:
grain_moisture_percent_tbl <- run_all("grain_moisture_percent", "grain_moisture_percent")  
write.xlsx(
  grain_moisture_percent_tbl,
  file = "statistics_tables/grain_moisture_percent_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

## 8- Statistics by Variety & Location & year

### 8.1- Cultivar-Level LSD and CV Analysis Across Locations and Years

In [117]:
# Define cultivar order for consistent reporting
cultivar_order <- c("CDC_Austenson", "AB_Hague", "Esma")


# Run LSD and CV analysis for one location-year combination
run_trait <- function(dat, location_, year_, outcome_col) {

  # Filter data and prepare cultivar and outcome variables
  df <- dat %>%
    dplyr::filter(location == location_, year == year_) %>%
    dplyr::transmute(
      location, year,
      cultivar = trimws(as.character(cultivar)),
      outcome  = as.numeric(.data[[outcome_col]])
    ) %>%
    dplyr::mutate(cultivar = factor(cultivar, levels = cultivar_order))

  # Return empty results if no data are available
  if (nrow(df) == 0) {
    return(list(
      lsd = tibble::tibble(
        cultivar = factor(cultivar_order, levels = cultivar_order),
        mean = numeric(), SE = numeric(), groups = character()
      ),
      cv  = tibble::tibble(sd = NA_real_, avg = NA_real_, cv = NA_real_)
    ))
  }

  # Compute coefficient of variation
  cv_tbl <- df %>%
    dplyr::summarise(
      sd  = stats::sd(outcome, na.rm = TRUE),
      avg = base::mean(outcome, na.rm = TRUE),
      cv  = sd / avg
    )

  # Run LSD test across cultivars with error handling
  lsd_tbl <- tryCatch({
    out <- df %>%
      dplyr::mutate(treatment = cultivar) %>%  # Use cultivar as LSD factor
      lsd_test_f2_01() %>%
      tibble::as_tibble()

    out$cultivar <- factor(trimws(as.character(out$treatment)), levels = cultivar_order)
    out$treatment <- NULL
    out
  }, error = function(e) {
    tibble::tibble(
      cultivar = factor(cultivar_order, levels = cultivar_order),
      mean = numeric(), SE = numeric(), groups = character()
    )
  })

  # Return LSD and CV results
  list(lsd = lsd_tbl, cv = cv_tbl)
}


# Define analysis plan (location × year combinations)
plan <- tibble::tribble(
  ~location,     ~year,
  "Falher",      2022,
  "Falher",      2023,
  "Lethbridge",  2022,
  "Lethbridge",  2023,
  "Lethbridge",  2024,
  "Vermilion",   2022,
  "Vermilion",   2023,
  "Vermilion",   2024
)


# Run cultivar analysis for one trait across all locations and years
run_all_cultivar <- function(outcome_col, trait_name = outcome_col, digits = 2) {

  # Apply run_trait to each location-year combination
  results <- plan %>%
    dplyr::mutate(res = purrr::pmap(
      list(location, year),
      ~ run_trait(cleaned_data_imputed, ..1, ..2, outcome_col)
    ))

  # Extract CV results
  cv_table <- results %>%
    dplyr::transmute(location, year, cv = purrr::map(res, "cv")) %>%
    tidyr::unnest(cv)

  # Extract LSD results
  lsd_table <- results %>%
    dplyr::transmute(location, year, lsd = purrr::map(res, "lsd")) %>%
    tidyr::unnest(lsd)

  # Merge results and format final output table
  lsd_table %>%
    dplyr::left_join(cv_table, by = c("location", "year")) %>%
    dplyr::mutate(
      trait = trait_name,
      cultivar = factor(trimws(as.character(cultivar)), levels = cultivar_order),
      mean_group_se = sprintf(
        paste0("%.", digits, "f %s \u00B1 %.", digits, "f"),
        mean, trimws(groups), SE
      )
    ) %>%
    dplyr::arrange(location, year, cultivar) %>%
    dplyr::select(
      location,
      year,
      cultivar,
      trait,
      mean,
      SE,
      groups,
      p.value,
      LSD,
      mean_group_se,
      sd,
      avg,
      cv
    )

}


### 8.2- Yield Kg/ha

In [118]:
variety_yield_adj_to_13_5_percent_results <- run_all_cultivar("yield_adj_to_13_5_percent", "yield_adj_to_13_5_percent", digits = 2)
write.xlsx(
  variety_yield_adj_to_13_5_percent_results,
  file = "statistics_tables/variety_yield_adj_to_13_5_percent_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.3- Yield Bu/ac

In [119]:
variety_yield_bu_results <- run_all_cultivar("yield_bu_acre", "yield_bu_acre", digits = 2)
write.xlsx(
  variety_yield_bu_results,
  file = "statistics_tables/variety_yield_bu_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.4- Lodging

In [120]:
variety_lodging_results <- run_all_cultivar("lodging_index_3", "lodging", digits = 2)
write.xlsx(
  variety_lodging_results,
  file = "statistics_tables/variety_lodging_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.5- TKW

In [121]:
variety_tkw_results <- run_all_cultivar("tkw", "tkw", digits = 2)
write.xlsx(
  variety_tkw_results,
  file = "statistics_tables/variety_tkw_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.6- Protein

In [122]:
variety_protein_results <- run_all_cultivar("protein_percent_dm", "protein", digits = 2)
write.xlsx(
  variety_protein_results,
  file = "statistics_tables/variety_protein_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.7- Average Plant height at maturity

In [123]:
variety_plant_height_results <- run_all_cultivar("avg_height_at_phys_maturity", "plant_height", digits = 2)
write.xlsx(
  variety_plant_height_results,
  file = "statistics_tables/variety_plant_height_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.8- Test weight

In [124]:
variety_test_weight_results <- run_all_cultivar("test_weight_kg_h_l", "test_weight", digits = 2)
write.xlsx(
  variety_test_weight_results,
  file = "statistics_tables/variety_test_weight_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.9- NDF

In [125]:
variety_ndf_results <- run_all_cultivar("ndf_percent_af", "ndf", digits = 2)
write.xlsx(
  variety_ndf_results,
  file = "statistics_tables/variety_ndf_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.10- IV_DEC

In [126]:
variety_iv_d_ec_percent_dm_results <- run_all_cultivar("iv_d_ec_percent_dm", "iv_d_ec_percent_dm", digits = 2)
write.xlsx(
  variety_iv_d_ec_percent_dm_results,
  file = "statistics_tables/variety_iv_d_ec_percent_dm_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.11- Starch

In [127]:
variety_starch_percent_dm_results <- run_all_cultivar("starch_percent_dm", "starch_percent_dm", digits = 2)
write.xlsx(
  variety_starch_percent_dm_results,
  file = "statistics_tables/variety_starch_percent_dm_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.12- DTM

In [128]:
variety_dtm_results <- run_all_cultivar("dtm", "dtm", digits = 2)
write.xlsx(
  variety_dtm_results,
  file = "statistics_tables/variety_dtm_results.xlsx",
  na.string = "",
  overwrite = TRUE
)

### 8.13- Grain moisture content

In [129]:
variety_grain_moisture_percent_results <- run_all_cultivar("grain_moisture_percent", "grain_moisture_percent", digits = 2)
write.xlsx(
  variety_grain_moisture_percent_results,
  file = "statistics_tables/variety_grain_moisture_percent_results.xlsx",
  na.string = "",
  overwrite = TRUE
)